# Parallel chunk review pilot

The frozen Notebook 06 development pilot contains 240 review units from 177 complete
parent documents. The first batch reviewed 12 units. This notebook assigns the other
228 units to 12 batches with disjoint parent rosters, and replays the returned decisions
against the versioned parent text and the 512-token reference budget.

The coordinator dispatches the assignments to Codex desktop windows in the same project.
Each window reads the existing shared packet file. Work can resume after a checkpoint
without reopening completed units. The notebook prepares assignments and validates
receipts locally. See [Codex window handoff](PREPROCESSING_WINDOW_HANDOFF.md).

## Publish and verify disjoint pilot batches

In [ ]:
import hashlib
import json
import tempfile
from collections import Counter
from pathlib import Path


def file_sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "notebooks/06_chunk_evaluation.ipynb").is_file()
)
BUNDLE = ROOT / ".cache/chunk-window-review/0a10356560c83672a868"
RUN = BUNDLE / "parallel-v1"
BATCH_COUNT = 12

source_manifest_path = BUNDLE / "MANIFEST.json"
source_manifest = read_json(source_manifest_path)
for name, expected_sha in source_manifest["files"].items():
    if file_sha256(BUNDLE / name) != expected_sha:
        raise ValueError(f"The Notebook 06 source changed: {name}.")
source_sha = source_manifest["files"]["PACKETS.jsonl"]
index = read_json(BUNDLE / "INDEX.json")
original = read_json(BUNDLE / "BATCH_001.json")

by_parent = {}
for item in index:
    by_parent.setdefault(item["release_row"], []).append(item)
already_assigned = {item["release_row"] for item in original["units"]}
if original["units"] != [item for item in index if item["release_row"] in already_assigned]:
    raise ValueError("The first batch did not contain its complete parent documents.")
outstanding = [units for parent, units in by_parent.items() if parent not in already_assigned]
identity = {
    "policy": "codex-parallel-pilot-v1",
    "source_manifest_sha256": file_sha256(source_manifest_path),
    "source_sha256": source_sha,
    "initial_batch_sha256": file_sha256(BUNDLE / "BATCH_001.json"),
    "batch_count": BATCH_COUNT,
}

if not RUN.exists():
    batches = [[] for _ in range(BATCH_COUNT)]
    unit_load = [0] * BATCH_COUNT
    for units in sorted(outstanding, key=lambda group: (-len(group), group[0]["line"])):
        lane = min(range(BATCH_COUNT), key=lambda i: (unit_load[i], len(batches[i]), i))
        batches[lane].extend(units)
        unit_load[lane] += len(units)
    with tempfile.TemporaryDirectory(prefix=".parallel-stage-", dir=BUNDLE) as temp:
        stage = Path(temp) / "snapshot"
        stage.mkdir(mode=0o700)
        (stage / "decisions").mkdir(mode=0o700)
        (stage / "dispatch").mkdir(mode=0o700)
        hashes = {}
        for ordinal, units in enumerate(batches, start=2):
            batch = f"batch-{ordinal:03d}"
            filename = f"BATCH_{ordinal:03d}.json"
            assignment = {
                "batch": batch,
                "source_file": "../PACKETS.jsonl",
                "source_sha256": source_sha,
                "units": sorted(units, key=lambda item: item["line"]),
                "receipt": f"decisions/{batch}.jsonl",
            }
            path = stage / filename
            path.write_text(json.dumps(assignment, indent=2) + "\n", encoding="utf-8")
            path.chmod(0o400)
            hashes[filename] = file_sha256(path)
        manifest = stage / "MANIFEST.json"
        manifest.write_text(
            json.dumps({**identity, "files": hashes}, indent=2) + "\n", encoding="utf-8"
        )
        manifest.chmod(0o400)
        if RUN.exists():
            raise FileExistsError("A pilot run appeared while assignments were staged.")
        stage.rename(RUN)

run_manifest = read_json(RUN / "MANIFEST.json")
if any(run_manifest.get(key) != value for key, value in identity.items()):
    raise ValueError("The published run belongs to another source or batch policy.")
assignment_names = {f"BATCH_{ordinal:03d}.json" for ordinal in range(2, BATCH_COUNT + 2)}
if set(run_manifest["files"]) != assignment_names:
    raise ValueError("The published run has an incomplete batch roster.")
assignments = {}
covered = set(already_assigned)
all_lines = {item["line"] for item in original["units"]}
for name, digest in sorted(run_manifest["files"].items()):
    path = RUN / name
    if file_sha256(path) != digest:
        raise ValueError(f"The assignment changed: {name}.")
    assignment = read_json(path)
    units = assignment["units"]
    batch_parents = {item["release_row"] for item in units}
    if (
        not units
        or assignment["source_file"] != "../PACKETS.jsonl"
        or assignment["source_sha256"] != source_sha
        or assignment["receipt"] != f"decisions/{assignment['batch']}.jsonl"
        or not covered.isdisjoint(batch_parents)
        or units != [item for item in index if item["release_row"] in batch_parents]
    ):
        raise ValueError(f"The batch roster is invalid: {name}.")
    covered.update(batch_parents)
    all_lines.update(item["line"] for item in units)
    assignments[assignment["batch"]] = assignment
if (
    covered != set(by_parent)
    or all_lines != set(range(1, source_manifest["units"] + 1))
    or len(assignments) != BATCH_COUNT
):
    raise ValueError("The assignments did not cover the complete frozen pilot.")
print(
    f"Verified {source_manifest['units']} units in {len(by_parent)} parents. "
    f"Published {len(assignments)} remaining batches covering "
    f"{sum(len(a['units']) for a in assignments.values())} units."
)
print(f"Pilot run: {RUN.relative_to(ROOT)}")


## Check progress without starting Spark

Run this cell again to see checkpointed decisions and final receipts. The replay
section verifies which returned boundaries pass the parent and token checks.

In [ ]:
import pandas as pd
from IPython.display import display

dispatch_path = RUN / "dispatch/STATE.json"
active_batches = (
    {f"batch-{item['batch']:03d}" for item in read_json(dispatch_path)["active"].values()}
    if dispatch_path.is_file()
    else set()
)
overview = []
for batch, assignment in assignments.items():
    final = RUN / assignment["receipt"]
    checkpoint = final.with_name(f".{batch}.partial.jsonl")
    state = "active" if batch in active_batches else "unassigned"
    committed = 0
    if final.is_file() or checkpoint.is_file():
        candidate = final if final.is_file() else checkpoint
        try:
            with candidate.open(encoding="utf-8") as handle:
                recorded = [json.loads(line)["packet_sha256"] for line in handle]
            expected_ids = {unit["packet_sha256"] for unit in assignment["units"]}
            if (
                len(set(recorded)) != len(recorded)
                or not set(recorded) <= expected_ids
                or (candidate == final and set(recorded) != expected_ids)
            ):
                state = "check_receipt"
            else:
                committed = len(recorded)
                state = "receipt_ready" if candidate == final else "checkpointed"
        except (ValueError, KeyError, FileNotFoundError):
            state = "check_receipt" if candidate == final else "checkpoint_unavailable"
    overview.append(
        {
            "Batch": batch,
            "State": state,
            "Units assigned": len(assignment["units"]),
            "Decisions on disk": committed,
        }
    )
display(pd.DataFrame(overview))


## Bind the original corpus and reference plan

Run the relevant local binding cells from Notebook 06 in a fresh project kernel. These
reuse the saved snapshots and check that the source file, parent text, and plan still
resolve to the version sent to the review windows.

In [ ]:
import nbformat
from pyspark.sql import functions as F

source_06 = nbformat.read(ROOT / "notebooks/06_chunk_evaluation.ipynb", as_version=4)
upstream = {cell.id: cell.source for cell in source_06.cells if cell.cell_type == "code"}
# Run the same input, packet and reference-budget checks used for the first assignment.
required_cells = [
    "06-configure",
    "06-bind",
    "06-review-queue",
    "06-review-protocol",
    "06-token-policy",
    "06-token-materialize",
    "06-token-review-gate",
    "06-window-sample",
    "06-window-bundle",
]
verified_06 = {"__name__": "__main__"}
for cell_id in required_cells:
    exec(compile(upstream[cell_id], f"06_chunk_evaluation.ipynb:{cell_id}", "exec"), verified_06)
if verified_06["window_dir"] != BUNDLE:
    raise ValueError("The verified Notebook 06 inputs resolved to a different bundle.")

parents = verified_06["parents"]
review_queue = verified_06["review_queue"]
token_plans = verified_06["token_plans"]
reference_review_parent = verified_06["reference_review_parent"]
make_review_packets = verified_06["make_review_packets"]
validate_reference_decisions = verified_06["validate_reference_decisions"]


## Replay returned decisions and report progress

A missing final receipt stays pending. A stale, duplicate or unavailable decision fails
validation. A semantic escalation or a parent that exceeds the token budget remains
open for follow-up. A complete pilot report records all outcomes, including holds.
Production decisions require a later exact cross-split check on the adopted boundaries.

In [ ]:
def validate_roster(assignment, decisions):
    expected_packets = {unit["packet_sha256"] for unit in assignment["units"]}
    received = {item["packet_sha256"]: item for item in decisions}
    if len(received) != len(decisions) or set(received) != expected_packets:
        raise ValueError("Each assigned packet must receive exactly one decision.")
    return received


def load_receipt(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


# Reject both a missing decision and a repeated decision before replaying any text.
fixture_units = {"units": [{"packet_sha256": "first"}, {"packet_sha256": "second"}]}
fixture_decisions = [{"packet_sha256": "first"}, {"packet_sha256": "second"}]
assert len(validate_roster(fixture_units, fixture_decisions)) == 2
for invalid in (fixture_decisions[:1], [fixture_decisions[0]] * 2):
    try:
        validate_roster(fixture_units, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("An incomplete or repeated receipt passed the roster check.")

all_assignments = {original["batch"]: (BUNDLE, original)}
all_assignments.update({key: (RUN, value) for key, value in assignments.items()})
ready = {}
batch_results = []
for batch, (directory, assigned) in all_assignments.items():
    path = directory / assigned["receipt"]
    if not path.is_file():
        batch_results.append({"batch": batch, "receipt_status": "pending"})
        continue
    decisions = load_receipt(path)
    received = validate_roster(assigned, decisions)
    ready[batch] = (assigned, received)
    batch_results.append(
        {
            "batch": batch,
            "receipt_status": "verified_roster",
            "units": len(received),
            "receipt_sha256": file_sha256(path),
        }
    )
print(f"Verified {len(ready)} of {len(all_assignments)} final receipt rosters.")

replay_parents = {
    item["release_row"] for assigned, _ in ready.values() for item in assigned["units"]
}
if replay_parents:
    selected = F.col("release_row").isin(*sorted(replay_parents))
    rows = {
        row.release_row: row
        for row in parents.where(selected)
        .select("release_row", "clean_text", "paragraphs")
        .collect()
    }
    queued = {row.release_row: row for row in review_queue.where(selected).collect()}
    packed = {
        row.release_row: row
        for row in token_plans.where(selected)
        .select("release_row", "clean_sha256", "plan")
        .collect()
    }
    if any(set(table) != replay_parents for table in (rows, queued, packed)):
        raise ValueError("The verified Notebook 06 inputs do not cover the review receipts.")

    parent_results = []
    decisions_by_status = Counter()
    decisions_by_tier = Counter()
    for batch, (assigned, received) in ready.items():
        by_release_row = {}
        for unit in assigned["units"]:
            by_release_row.setdefault(unit["release_row"], []).append(unit)
            outcome = received[unit["packet_sha256"]]["status"]
            decisions_by_status[outcome] += 1
            decisions_by_tier[unit["tier"], outcome] += 1
        for release_row, units in by_release_row.items():
            text = rows[release_row]
            plan = reference_review_parent(queued[release_row], packed[release_row])
            expected_packets = {
                packet["packet_sha256"]
                for packet in make_review_packets(plan, text.clean_text, text.paragraphs)
            }
            if expected_packets != {unit["packet_sha256"] for unit in units}:
                raise ValueError("The receipt no longer matches the parent review windows.")
            verified = validate_reference_decisions(
                plan, text.clean_text, text.paragraphs,
                [received[unit["packet_sha256"]] for unit in units],
            )
            parent_results.append(
                {
                    "batch": batch,
                    "release_row": release_row,
                    "tier": units[0]["tier"],
                    "status": verified["status"],
                    "units": len(units),
                    "body_sha256": plan.clean_sha256,
                    "reference_plan_sha256": plan.plan_sha256,
                    "reviewed_chunk_ends_in_parent": verified["ends"],
                    "reference_token_counts": verified.get("reference_token_counts"),
                }
            )
    parent_status = Counter(row["status"] for row in parent_results)
    if len(parent_results) != len(replay_parents):
        raise ValueError("A final parent was omitted or reviewed by multiple pilot batches.")
    display(pd.DataFrame(decisions_by_status.items(), columns=["Unit decision", "Count"]))
    display(pd.DataFrame(parent_status.items(), columns=["Parent status", "Count"]))
    display(
        pd.DataFrame(
            [(*key, count) for key, count in sorted(decisions_by_tier.items())],
            columns=["Tier", "Decision", "Units"],
        )
    )
else:
    parent_results = []
    decisions_by_status = Counter()
    parent_status = Counter()

if len(ready) == len(all_assignments):
    if len(parent_results) != source_manifest["parents"]:
        raise ValueError("The pilot did not reconcile the complete frozen roster.")
    results = {
        "source_sha256": source_sha,
        "assignment_manifest_sha256": file_sha256(RUN / "MANIFEST.json"),
        "reviewed_parents": len(parent_results),
        "reviewed_units": sum(decisions_by_status.values()),
        "decisions_by_status": dict(decisions_by_status),
        "parents_by_status": dict(parent_status),
        "receipts": batch_results,
        "parents": sorted(parent_results, key=lambda item: item["release_row"]),
    }
    audit_path = RUN / "PILOT_REPLAY.json"
    if audit_path.exists():
        if read_json(audit_path) != results:
            raise ValueError("The final replay differs from the published audit.")
    else:
        with tempfile.TemporaryDirectory(prefix=".verify-pilot-", dir=RUN) as temp:
            path = Path(temp) / audit_path.name
            path.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
            path.chmod(0o600)
            path.rename(audit_path)
    print(f"Complete pilot replay: {audit_path.relative_to(ROOT)}")


## Measure the run and verify the handoff checkpoint

In [ ]:
from datetime import UTC, datetime

dispatch_path = RUN / "dispatch/STATE.json"
if dispatch_path.is_file():
    progress = read_json(dispatch_path)
    finished_attempts = [
        (batch, attempt)
        for batch, attempts in progress["attempts"].items()
        for attempt in attempts
        if attempt.get("turn", {}).get("completedAt")
    ]
    records = [
        {
            "Batch": batch,
            "Attempt": attempt["kind"],
            "Codex window": attempt["threadId"],
            "Seconds": round(attempt["turn"]["durationMs"] / 1000, 1),
            "Units checkpointed": attempt.get("retained"),
        }
        for batch, attempt in sorted(
            finished_attempts, key=lambda row: (int(row[0]), row[1]["turn"]["startedAt"])
        )
    ]
    if records:
        display(pd.DataFrame(records))
        start = progress["firstDispatchAt"]
        end = max(attempt["turn"]["completedAt"] for _, attempt in finished_attempts)
        print(f"First dispatch: {datetime.fromtimestamp(start, UTC).isoformat()}")
        print(f"Last completed attempt: {datetime.fromtimestamp(end, UTC).isoformat()}")
        elapsed = end - start
        print(f"Elapsed review window span: {elapsed / 60:.1f} minutes")
        if len(ready) == len(all_assignments) and elapsed > 0:
            new_units = sum(len(received) for batch, (_, received) in ready.items()
                            if batch != original["batch"])
            review_seconds = sum(attempt["turn"]["durationMs"] / 1000
                                 for _, attempt in finished_attempts)
            print(f"Parallel throughput: {3600 * new_units / elapsed:.1f} units per wall hour")
            units_per_turn_hour = 3600 * new_units / review_seconds
            print(f"Reviewer throughput: {units_per_turn_hour:.1f} units per turn hour")
    relay = progress["handoff"]
    if relay.get("to") and "batch-002" in ready:
        completed = ready["batch-002"][1]
        checkpoint_path = RUN / "dispatch/RELAY_002_CHECKPOINT.jsonl"
        if file_sha256(checkpoint_path) != relay["checkpointSha"]:
            raise ValueError("The retained handoff checkpoint changed.")
        checkpoint = load_receipt(checkpoint_path)
        if len(checkpoint) != relay["retained"] or any(
            completed[item["packet_sha256"]] != item for item in checkpoint
        ):
            raise ValueError("The receiving window did not preserve the checkpoint decisions.")
        print(
            f"Cross-window handoff verified with {len(checkpoint)} retained decisions "
            f"and {len(completed) - len(checkpoint)} remaining decisions."
        )
